# Main file 

05.11.2025

Written by: Arnav Kumar Upadhyay, Cedric Meyer adapted from MATLAB code of Marc, Jan Spindler DOI-https://doi.org/10.1021/acssynbio.4c00578

This script is the main script for the TXTL v2 model in INDEPENDENT GENE SIMULATION mode.

The script simulates each gene independently without competition for shared resources.


## Settings

In [4]:
# Packages functions and data types --------
include("packages.jl") 


# Sequences 
sequence_file	= "lacS.fa"
hybrid_file		= "lacS.txt"

# initialization of data containers
scale = txtl_types.init_scale()         # struct that makes it possible to influence rates by scaling them
# tags = txtl_types.init_data()           # data container saving everything necessary to start the ODE solver 
µ = 1                    # A growth rate (0.1 to 1.8) can be chosen and the proteins of the TXTL machinery will be tuned according to that growth rate and the processivity of the RNA polymerase


## flags for efficiency factors
positive_aminoacid	= true 
anti_shinedalgarno	= true      # If no hybridization file is available put flag to false. We apply the intaRNA program’s intarna function (q: aSD, t: sequence, SeedBP: 5, out: tMinE) to generate a hybridization profile along the sequence. Link to the website: https://rna.informatik.uni-freiburg.de/IntaRNA/Input.jsp;jsessionid=2AE257E91E7276283F22CD77A5A8AA03
dynamic_efficiency	= true 

flags = txtl_types.flags(positive_aminoacid, anti_shinedalgarno, dynamic_efficiency)

## Import information about model species and sequences 
seqs, seq_names, hyb_file_dict, species_and_seqs, idx = species_and_sequences(µ, sequence_file, hybrid_file)

## Calculate efficiency factors 
efac_pos_aa_dict, efac_int_sd_dict, dynamic_efficiencies = efficiency_factor_initialization(seqs, seq_names, hyb_file_dict, species_and_seqs)


## calculation of initial concentrations
y0, initial_concentrations = calculation_of_initial_concentration(species_and_seqs, seqs, seq_names, scale, idx);


## calculate and load kinetic parameters
p_met, p_seq = parameters(initial_concentrations, species_and_seqs);


## Release factor balance
RF_balance!(y0, initial_concentrations)


# Preintegration
## Definition of mass matrix
conserved = [idx.EFG, idx.EFD] # EFG, EFTu_GDP - Conserved species, Algebraic equations are given by mass balances
M = Matrix{Float64}(I, length(y0), length(y0)) # Creates unity matrix in the size of y0
M[conserved, :] .= 0; # setting mass matrix to zero to  remove time derivatives from those variables. This converts these differential equations into algebraic constraints

tags_pre = txtl_types.data_struct_preintegration(scale, idx, species_and_seqs, flags, dynamic_efficiencies, initial_concentrations, p_met, p_seq)

## Integration of the DAE system without DNA in the system 
y_steady_state = preintegration(tags_pre, y0, M, scale);


# Elongation factor balance 
EF_balance!(tags_pre, y0, y_steady_state)


# Build states vector for multiple sequences 
y0_all = combined_states_vector(tags_pre, y_steady_state, seqs, seq_names)

# Prepare per-sequence tags (for use in ODE)
local_tags_list = generate_local_tags(seqs, seq_names, tags_pre, efac_pos_aa_dict, efac_int_sd_dict);


✅ Successfully loaded 1 sequences


LoadError: type data_struct_preintegration has no field Seq

In [9]:
tags_pre.species_and_sequences.codAS

38-element Vector{String}:
 "GCU, GCA, GCG"
 "GCC,"
 "UGC, UGU,"
 "GAC, GAU,"
 "GAA, GAG,"
 "UUC, UUU,"
 "GGA, GGG,"
 "GGC, GGU,"
 "CAC, CAU, "
 "AUC, AUU, AUA "
 "AAA, AAG,"
 "CUG,"
 "CUC, CUU,"
 ⋮
 "AGG,"
 "UCA, UCU, UCG"
 "UCG,"
 "AGC, AGU,"
 "UCC, UCU,"
 "ACC, ACU,"
 "ACG,"
 "ACA, ACU, ACG"
 "GUA, GUG, GUU"
 "GUC, GUU,"
 "UGG,"
 "UAC, UAU,"

In [ ]:



# Integration
M = Matrix{Float64}(I, length(y0_all), length(y0_all))
M[conserved, :] .= 0

tspan 	= (0.0, 100.0) ## Define time span for the simulation
p_parallel = (local_tags_list,seq_names,seqs)
f 		= ODEFunction(ODE_parallel_distributed!, mass_matrix = M) ## Define ODEFunction to make use of mass matrix
prob 	= ODEProblem(f, y0_all, tspan, p_parallel) ## Create an ODE problem

# --- Timing the integration step ---
println("\n⏳ Integrating ODE system (distributed, test set)...")


In [2]:

@benchmark sol = solve(prob, FBDF(autodiff=false), reltol = 1e-10, abstol = 1e-6) ## Solve ODE system

BenchmarkTools.Trial: 7 samples with 1 evaluation per sample.
 Range (min … max):  725.677 ms … 808.101 ms  ┊ GC (min … max): 17.68% … 16.92%
 Time  (median):     763.302 ms               ┊ GC (median):    16.76%
 Time  (mean ± σ):   771.822 ms ±  28.621 ms  ┊ GC (mean ± σ):  16.03% ±  1.84%

  ▁                       ▁  █           ▁                  ▁ ▁  
  █▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁█▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁█ ▁
  726 ms           Histogram: frequency by time          808 ms <

 Memory estimate: 1.14 GiB, allocs estimate: 16584904.

# Profiling

In [5]:
using Profile, ProfileView
@profile solve(prob, FBDF(autodiff=false), reltol = 1e-10, abstol = 1e-6)


retcode: Success
Interpolation: 3rd order Hermite
t: 231-element Vector{Float64}:
   0.0
   1.6392854578321727e-9
   7.858580010350924e-9
   1.997491565678608e-8
   3.689305711281868e-8
   1.5049488924448925e-7
   3.8727422627369924e-7
   6.765961266888345e-7
   9.6591802710397e-7
   1.723018633538079e-6
   2.6837433070101694e-6
   3.64446798048226e-6
   4.714537043677668e-6
   ⋮
   1.0067260227230634
   1.1272705959874623
   1.2738743809087802
   1.4746545720139381
   1.7439638463645946
   2.1302062587076303
   2.801129715941944
   4.629434646003425
   8.81553886059708
  19.355338974617542
  53.22216314585782
 100.0
u: 231-element Vector{Vector{Float64}}:
 [20.066985600703717, 20.086056744926683, 0.011764823802035312, 0.020942576143074654, 2.3889195399692102, 1.5446822292047273, 0.6637432999699595, 0.7463944193090334, 0.21061805577608217, 2.3619242296179084  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [20.066985600703717, 20.086056744926683, 0.011764823802035312, 0.020942576

In [6]:
ProfileView.view()


Gtk4.GtkWindowLeaf(accessible-role=GTK_ACCESSIBLE_ROLE_WINDOW, name="", parent, root, width-request=-1, height-request=-1, visible=true, sensitive=true, can-focus=true, has-focus=false, can-target=true, focus-on-click=true, focusable=false, has-default=false, receives-default=false, cursor, has-tooltip=false, tooltip-markup=nothing, tooltip-text=nothing, opacity=1,000000, overflow=GTK_OVERFLOW_HIDDEN, halign=GTK_ALIGN_FILL, valign=GTK_ALIGN_FILL, margin-start=0, margin-end=0, margin-top=0, margin-bottom=0, hexpand=false, vexpand=false, hexpand-set=false, vexpand-set=false, scale-factor=1, css-name="window", css-classes, layout-manager, limit-events=false, title="Profile", resizable=true, modal=false, default-width=800, default-height=600, destroy-with-parent=false, hide-on-close=false, icon-name=nothing, display, decorated=true, deletable=true, transient-for, application, default-widget, focus-widget, child, titlebar, handle-menubar-accel=true, is-active=true, suspended=false, startup-

.\compiler\abstractinterpretation.jl:2282, MethodInstance for Core.Compiler.abstract_call(::Core.Compiler.NativeInterpreter, ::Core.Compiler.ArgInfo, ::Core.Compiler.StmtInfo, ::Core.Compiler.InferenceState, ::Int64)


# Check type stability

In [ ]:
using Test
du = similar(y0_all)
du = fill!(du, 0.0);
u = y0_all;
p = (local_tags_list,seq_names,seqs)
t = tspan;

In [70]:
ODE_parallel_distributed!(du, u, p, t)

## Type stability in ODE_parallel_distributed! Start

In [69]:
@code_warntype ODE_parallel_distributed_1(du, u, p, t)


MethodInstance for ODE_parallel_distributed_1(::Vector{Float64}, ::Vector{Float64}, ::Tuple{Vector{Main.txtl_types.data_struct}, Vector{String}, Dict{String, Main.txtl_types.SequenceData}}, ::Tuple{Float64, Float64})
  from ODE_parallel_distributed_1(du, u, p, t) @ Main In[62]:1
Arguments
  #self#::Core.Const(Main.ODE_parallel_distributed_1)
  du::Vector{Float64}
  u::Vector{Float64}
  p::Tuple{Vector{Main.txtl_types.data_struct}, Vector{String}, Dict{String, Main.txtl_types.SequenceData}}
  t::Tuple{Float64, Float64}
Locals
  @_6::Union{Nothing, Tuple{Int64, Int64}}
  @_7::Union{Nothing, Tuple{Int64, Int64}}
  #60::var"#60#62"{Tuple{Float64, Float64}, Vector{Vector{Float64}}, Vector{Float64}, Vector{Main.txtl_types.data_struct}}
  #59::var"#59#61"{Int64}
  @_10::Union{Nothing, Tuple{Int64, Int64}}
  du_metab::Vector{Float64}
  results::Vector{Vector{Float64}}
  du_codons::Vector{Vector{Float64}}
  du_metab_all::Vector{Vector{Float64}}
  n_seq::Int32
  y_codons_lengths::Vector{Int64}
 

In [72]:
@benchmark sol = solve(prob, FBDF(autodiff=false), reltol = 1e-10, abstol = 1e-6) ## Solve ODE system


BenchmarkTools.Trial: 3 samples with 1 evaluation per sample.
 Range (min … max):  1.802 s …    2.119 s  ┊ GC (min … max): 10.97% … 16.98%
 Time  (median):     1.810 s               ┊ GC (median):    12.80%
 Time  (mean ± σ):   1.910 s ± 180.714 ms  ┊ GC (mean ± σ):  13.77% ±  3.08%

  ██                                                       █  
  ██▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  1.8 s          Histogram: frequency by time         2.12 s <

 Memory estimate: 1.05 GiB, allocs estimate: 16545200.

# Allocations

Aim: Reduce to zero allocations.

## ODE_parallel_distributed

In [2]:
du = similar(y0_all)
du = fill!(du, 0.0);
u = y0_all;
p = (local_tags_list,seq_names,seqs)
t = tspan;

In [41]:
@btime ODE_parallel_distributed!($du, $u, $p, $t) setup=(du=copy(du), u=copy(u)) evals=1 samples=1


  689.100 μs (11561 allocations: 825.75 KiB)


In [111]:
@benchmark sol = solve(prob, FBDF(autodiff=false), reltol = 1e-10, abstol = 1e-6) ## Solve ODE system

BenchmarkTools.Trial: 3 samples with 1 evaluation per sample.
 Range (min … max):  1.880 s …    2.347 s  ┊ GC (min … max): 12.74% … 15.29%
 Time  (median):     1.916 s               ┊ GC (median):    12.50%
 Time  (mean ± σ):   2.048 s ± 260.123 ms  ┊ GC (mean ± σ):  13.25% ±  2.04%

  █   █                                                    █  
  █▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  1.88 s         Histogram: frequency by time         2.35 s <

 Memory estimate: 1.05 GiB, allocs estimate: 16531020.

## rhs_metabolites

In [43]:
function test_ODE(du, u, p, t)

    local_tags_list = p[1]				# Extraction of local_tags_list from p
    seq_names       = p[2]
    seqs            = p[3]
    tags            = local_tags_list[1]

    n_metab = length(tags.Tag) -5 		# number of metabolites in the shared pool that is used by all sequences 

    @views y_metab =  u[1:n_metab]				# Save all pool species from u in a vector 
    idx = n_metab						# Index of last pool species
   
    y_codons_lengths = Int[length(seqs[seq_name].Codon) + 4 for seq_name in seq_names]	# List with sequence lengths - 1 stop codon + 5 observable variables

    y_codons = Vector{SubArray{Float64,1,typeof(u),Tuple{UnitRange{Int64}},true}}(undef, length(y_codons_lengths))
    idx = n_metab
    @views for (i,L) in enumerate(y_codons_lengths)
        y_codons[i] = u[idx+1:idx+L]  # ✅ View statt Kopie
        idx += L
    end

    
    n_seq = tags.no_of_seqs 		# save number of sequences
    du_metab_all = [zeros(n_metab) for _ in 1:n_seq]	# List to store a du vector for the metabolite pool for every sequence 
    du_codons = Vector{Vector{Float64}}(undef, n_seq)	# List to store a du vector for the codons for every sequence 

    # Distributed parallelization of per-sequence rate calculations
    i = 1
    local_tags = deepcopy(local_tags_list[i])		# Extraction of local tags
    u_seq = vcat(y_metab, y_codons[i])				# Sequenzspezifischer state vector
    du_seq = zeros(length(u_seq))					# Sequenzspezifischer rates vector

    return du_seq, u_seq, local_tags

end

du = similar(y0_all)
du = fill!(du, 0.0);
u = y0_all;
p = (local_tags_list,seq_names,seqs)
t = tspan;

du, u, p = test_ODE(du, u, p, t)


([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [20.066985600703717, 20.086056744926683, 0.011764823802035312, 0.020942576143074654, 2.3889195399692102, 1.5446822292047273, 0.6637432999699595, 0.7463944193090334, 0.21061805577608217, 2.3619242296179084  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], Main.txtl_types.data_struct("", 38, 1.0, Main.txtl_types.scale_struct(1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], 0.0, 0.0), 1, 1.0, ["S30", "S50", "IF1", "IF2", "IF3", "SC1", "SC2", "SC3", "SC4", "PIC"  …  "t4t", "v1t", "v2t", "w1t", "y1t", "RNA1", "S701", "rna1", "PRT1", "PTC1"], LongSequence{RNAAlphabet{2}}[AUG, ACC, AUG, AUU, ACG, GAU, UCA, CUG, GCC, GUC  …  CAA, CUU, AAU, CGC, CUU, GCA, GCA, CAU, CCC, UAA], ["lacS"], ["S30", "S50", "IF1", "IF2", "IF3", "SC1", "SC2", "SC3", "SC4", "PIC"  …  "s2t", "s3t", "s5t", "t1t", "t2t", "

In [44]:
@btime rhs_metabolites!($du, $u, $p) setup=(du=copy(du), u=copy(u)) evals=1 samples=1

  142.200 μs (2168 allocations: 34.86 KiB)


## rhs sequence

In [3]:
function test_ODE(du, u, p, t)

    local_tags_list = p[1]				# Extraction of local_tags_list from p
    seq_names       = p[2]
    seqs            = p[3]
    tags            = local_tags_list[1]

    n_metab = length(tags.Tag) -5 		# number of metabolites in the shared pool that is used by all sequences 

    @views y_metab =  u[1:n_metab]				# Save all pool species from u in a vector 
    idx = n_metab						# Index of last pool species
   
    y_codons_lengths = Int[length(seqs[seq_name].Codon) + 4 for seq_name in seq_names]	# List with sequence lengths - 1 stop codon + 5 observable variables

    y_codons = Vector{SubArray{Float64,1,typeof(u),Tuple{UnitRange{Int64}},true}}(undef, length(y_codons_lengths))
    idx = n_metab
    @views for (i,L) in enumerate(y_codons_lengths)
        y_codons[i] = u[idx+1:idx+L]  # ✅ View statt Kopie
        idx += L
    end

    
    n_seq = tags.no_of_seqs 		# save number of sequences
    du_metab_all = [zeros(n_metab) for _ in 1:n_seq]	# List to store a du vector for the metabolite pool for every sequence 
    du_codons = Vector{Vector{Float64}}(undef, n_seq)	# List to store a du vector for the codons for every sequence 

    # Distributed parallelization of per-sequence rate calculations
    i = 1
    local_tags = deepcopy(local_tags_list[i])		# Extraction of local tags
    u_seq = vcat(y_metab, y_codons[i])				# Sequenzspezifischer state vector
    du_seq = zeros(length(u_seq))					# Sequenzspezifischer rates vector

    return du_seq, u_seq, local_tags

end

du = similar(y0_all)
du = fill!(du, 0.0);
u = y0_all;
p = (local_tags_list,seq_names,seqs)
t = tspan;

du, u, p = test_ODE(du, u, p, t)

rhs_metabolites!(du, u, p)


In [47]:
@btime rhs_sequence!($du, $u, $p) setup=(du=copy(du), u=copy(u)) evals=1 samples=1

  314.600 μs (6989 allocations: 395.31 KiB)


170-element Vector{Float64}:
 -4.547473508864641e-13
  0.0
 -2.525757381022231e-14
 -2.5579538487363607e-13
 -4.547473508864641e-13
  1.1368683772161603e-13
  8.526512829121202e-14
  3.304023721284466e-13
  9.631184738623233e-14
 -1.7053025658242404e-13
  0.0
  0.0
  0.0
  ⋮
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0

## Index dict to index struct

In [48]:
@benchmark sol = solve(prob, FBDF(autodiff=false), reltol = 1e-10, abstol = 1e-6) ## Solve ODE system

BenchmarkTools.Trial: 7 samples with 1 evaluation per sample.
 Range (min … max):  676.043 ms … 877.566 ms  ┊ GC (min … max): 17.78% … 15.20%
 Time  (median):     698.899 ms               ┊ GC (median):    17.53%
 Time  (mean ± σ):   721.429 ms ±  70.124 ms  ┊ GC (mean ± σ):  17.15% ±  0.98%

  ▁  █  ▁  ▁  ▁                                               ▁  
  █▁▁█▁▁█▁▁█▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  676 ms           Histogram: frequency by time          878 ms <

 Memory estimate: 1.13 GiB, allocs estimate: 16509560.

# Avoid divisions

In [51]:
@benchmark sol = solve(prob, FBDF(autodiff=false), reltol = 1e-10, abstol = 1e-6) ## Solve ODE system

BenchmarkTools.Trial: 7 samples with 1 evaluation per sample.
 Range (min … max):  692.805 ms … 810.480 ms  ┊ GC (min … max): 17.53% … 15.52%
 Time  (median):     715.411 ms               ┊ GC (median):    17.59%
 Time  (mean ± σ):   727.942 ms ±  40.495 ms  ┊ GC (mean ± σ):  17.64% ±  1.10%

  █ █    █   █      █       █                                 █  
  █▁█▁▁▁▁█▁▁▁█▁▁▁▁▁▁█▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  693 ms           Histogram: frequency by time          810 ms <

 Memory estimate: 1.13 GiB, allocs estimate: 16509560.

# Analysis of rhs_sequence as it has 7000 allocations

Plan:
- Divide the function into smaller functions and benchmark those with @btime

In [4]:
function rhs_sequence1(du, u, p)

        # du = zeros(Float64, length(u))
    tags = p
    # Unpack parameters 
    V_degmax, k_cat_Pol, k_D, C_RNP, V_TCmax, K_M_D, C_RNE, k_cat_E, K_M_deg, k_TLEmaxj, KM_T3j, KM_EFGGTP, KM_RK, KM_RF3GTP, k_RF1_diss_GTP, k_RF3_diss_GDP, k_GTP_hydr, k_RF1_ass, k_RF3_GDP_ass, k_RF3_GTP_ass, k_RF3_GDP_diss, k_RF3_GTP_diss, A1, A2, A3, A4, A5, k_cat_K_M_G1, k_cat_K_M_G2, k_RRF, q_RRF, one_q_G2_k_split, k_TE = tags.p_seq

    # Unpack constant variables 
    ATP, GTP, UTP, CTP, ADP, GDP, UDP, CDP, AMP, GMP, UMP, CMP, Eut, Est, RFt, RRF, Ala, Cys, Asp, Glu, Phe, Gly, His, Ile, Lys, Leu, Met, Asn, Pro, Gln, Arg, Ser, Thr, Val, Trp, Tyr, C_D = tags.constant_vars 

    # Unpack dynamic variables 
    S30, S50, IF1, IF2, IF3, SC1, SC2, SC3, SC4, PIC, EFG, EFD, EGT, EGD, EFT, RF3, RGT, RGD, b1t, ta1, ta2, tc1, td1, te2, tf1, tg2, tg3, th1, ti1, tk1, t1l, t2l, t3l, t4l, t5l, tm1, tn1, tp1, tp2, tp3, tq1, tq2, tr2, tr3, tr4, tr5, ts1, ts2, ts3, ts5, tt1, tt2, tt4, tv1, tv2, tw1, ty1, tA1, tA2, tC1, tD1, tE2, tF1, tG2, tG3, tH1, tI1, tK1, tL1, tL2, tL3, tL4, tL5, tM1, tN1, tP1, tP2, tP3, tQ1, tQ2, tR2, tR3, tR4, tR5, tS1, tS2, tS3, tS5, tT1, tT2, tT4, tV1, tV2, tW1, tY1, a1t, a2t, c1t, d1t, e2t, f1t, g2t, g3t, h1t, i1t, k1t, l1t, l2t, l3t, l4t, l5t, m1t, n1t, p1t, p2t, p3t, q1t, q2t, r2t, r3t, r4t, r5t, s1t, s2t, s3t, s5t, t1t, t2t, t4t, v1t, v2t, w1t, y1t, RNA, S70, rna, PRT, PTC = u 

    # Containers 
    num_species = length(tags.Name)
    len_sequence = length(tags.Seq) # Codon length with stop codon

    # Define RIB1
    idx_first_codon = length(tags.Tag)+1
    rate_RIB        = zeros(Float64, len_sequence-1);
    @views c_RIB	 		= u[idx_first_codon:end]

    n_DNA_t = 1;
    n_DNA_s = 1;

    # # # tQSSR model used in Spindler

    # V_TCmax 	= k_cat_Pol * C_RNP
    # V_degmax    = k_cat_E * C_RNE

    du[tags.idx.RNA] += (V_TCmax/(len_sequence*3) * C_D * tags.scale.V_TC_scale / ( K_M_D * (1 + (n_DNA_t-n_DNA_s)* 0.001/K_M_D)+ C_RNP + C_D * tags.scale.V_TC_scale )) - 
    (V_degmax * RNA / (K_M_deg  + C_RNE + RNA ))	# 170


    # println(du[tags.idx.RNA])

    return nothing 

end

rhs_sequence1 (generic function with 1 method)

In [12]:
tags.p_seq

33-element Vector{Any}:
     9.739794291412766
  2673.174
     0.289
     7.8527540921213745
 20991.778067452462
     0.0063
     0.8696244903047113
    11.2
     0.5599999999999999
  1440.0
     0.4
     0.22
     0.0083
     ⋮
  1440.0
     0.041
     1.07
     0.126
     0.137
     0.538
  3780.0
   486.0
   900.0
  3480.0
     1.1
  1440.0

In [8]:
@code_warntype rhs_sequence1(du, u, p)

MethodInstance for rhs_sequence1(::Vector{Float64}, ::Vector{Float64}, ::Main.txtl_types.data_struct)
  from rhs_sequence1(du, u, p) @ Main In[4]:1
Arguments
  #self#::Core.Const(Main.rhs_sequence1)
  du::Vector{Float64}
  u::Vector{Float64}
  p::Main.txtl_types.data_struct
Locals
  @_5::Int64
  @_6::Int64
  @_7::Int64
  n_DNA_s::Int64
  n_DNA_t::Int64
  c_RIB::SubArray{Float64, 1, Vector{Float64}, Tuple{UnitRange{Int64}}, true}
  rate_RIB::Vector{Float64}
  idx_first_codon::Int64
  len_sequence::Int64
  num_species::Int64
  PTC::Float64
  PRT::Float64
  rna::Float64
  S70::Float64
  RNA::Float64
  y1t::Float64
  w1t::Float64
  v2t::Float64
  v1t::Float64
  t4t::Float64
  t2t::Float64
  t1t::Float64
  s5t::Float64
  s3t::Float64
  s2t::Float64
  s1t::Float64
  r5t::Float64
  r4t::Float64
  r3t::Float64
  r2t::Float64
  q2t::Float64
  q1t::Float64
  p3t::Float64
  p2t::Float64
  p1t::Float64
  n1t::Float64
  m1t::Float64
  l5t::Float64
  l4t::Float64
  l3t::Float64
  l2t::Float64
  l1t:

In [5]:
@btime rhs_sequence1($du, $u, $p) setup=(du=copy(du), u=copy(u)) evals=1 samples=1

  12.700 μs (22 allocations: 624 bytes)


In [ ]:
function rhs_sequence!(du, u, p)


    # du = zeros(Float64, length(u))
    tags = p
    # Unpack parameters 
    k_cat_Pol, k_D, C_RNP, V_TCmax, K_M_D, C_RNE, k_cat_E, K_M_deg, k_TLEmaxj, KM_T3j, KM_EFGGTP, KM_RK, KM_RF3GTP, k_RF1_diss_GTP, k_RF3_diss_GDP, k_GTP_hydr, k_RF1_ass, k_RF3_GDP_ass, k_RF3_GTP_ass, k_RF3_GDP_diss, k_RF3_GTP_diss, A1, A2, A3, A4, A5, k_cat_K_M_G1, k_cat_K_M_G2, k_RRF, q_RRF, one_q_G2_k_split, k_TE = tags.p_seq

    # Unpack constant variables 
    ATP, GTP, UTP, CTP, ADP, GDP, UDP, CDP, AMP, GMP, UMP, CMP, Eut, Est, RFt, RRF, Ala, Cys, Asp, Glu, Phe, Gly, His, Ile, Lys, Leu, Met, Asn, Pro, Gln, Arg, Ser, Thr, Val, Trp, Tyr, C_D = tags.constant_vars 

    # Unpack dynamic variables 
    S30, S50, IF1, IF2, IF3, SC1, SC2, SC3, SC4, PIC, EFG, EFD, EGT, EGD, EFT, RF3, RGT, RGD, b1t, ta1, ta2, tc1, td1, te2, tf1, tg2, tg3, th1, ti1, tk1, t1l, t2l, t3l, t4l, t5l, tm1, tn1, tp1, tp2, tp3, tq1, tq2, tr2, tr3, tr4, tr5, ts1, ts2, ts3, ts5, tt1, tt2, tt4, tv1, tv2, tw1, ty1, tA1, tA2, tC1, tD1, tE2, tF1, tG2, tG3, tH1, tI1, tK1, tL1, tL2, tL3, tL4, tL5, tM1, tN1, tP1, tP2, tP3, tQ1, tQ2, tR2, tR3, tR4, tR5, tS1, tS2, tS3, tS5, tT1, tT2, tT4, tV1, tV2, tW1, tY1, a1t, a2t, c1t, d1t, e2t, f1t, g2t, g3t, h1t, i1t, k1t, l1t, l2t, l3t, l4t, l5t, m1t, n1t, p1t, p2t, p3t, q1t, q2t, r2t, r3t, r4t, r5t, s1t, s2t, s3t, s5t, t1t, t2t, t4t, v1t, v2t, w1t, y1t, RNA, S70, rna, PRT, PTC = u 

    # Containers 
    num_species = length(tags.Name)
    len_sequence = length(tags.Seq) # Codon length with stop codon

    # Define RIB1
    idx_first_codon = length(tags.Tag)+1
    rate_RIB        = zeros(Float64, len_sequence-1);
    c_RIB	 		= u[idx_first_codon:end]

    # Transcription -----------------

    # Free mRNA (RNA) levels are given by a starting value to implement mRNA concentrations from transcriptomics measurements

    n_DNA_t = 1;
    n_DNA_s = 1;

    # # # tQSSR model used in Spindler

    V_TCmax 	= k_cat_Pol * C_RNP
    V_degmax    = k_cat_E * C_RNE

    du[tags.idx.RNA] += (V_TCmax/(len_sequence*3) * C_D * tags.scale.V_TC_scale / ( K_M_D * (1 + (n_DNA_t-n_DNA_s)* 0.001/K_M_D)+ C_RNP + C_D * tags.scale.V_TC_scale )) - 
    (V_degmax * RNA / (K_M_deg  + C_RNE + RNA ))	# 170

    ## translation elongation ----------------------------------------

    #= queuing factor q_R_elo
        Nieß 2017:
        Any influence of ribosomal hindrance was incorporated by applying a queuing factor q_R_elo,
        which reduces the elongation rate with increasing ribosome density. For an elongation step from codon 
        j to j+1, the queuing factor corresponds to the density of ribosomes on  the 12 codons further downstream. =#         

    total_RNA   = RNA + sum(c_RIB[1:12])

    q_R_elo     = ones(len_sequence-1);
        
    for i in 2:(len_sequence - 13)

        if total_RNA-sum(c_RIB[i:i+12]) == 0
            q_R_elo[i] 	= 0;
        else
            q_R_elo[i] 	= 1 - c_RIB[i+12] / (total_RNA-sum(c_RIB[i:i+11]));
        
        end
    end 

        
    for i in (len_sequence-13) : (len_sequence-1)
        if total_RNA - sum(c_RIB[i : end]) == 0
            q_R_elo[i] = 0
        else 
            q_R_elo[i] = 1 - (c_RIB[end] + PTC) / (total_RNA - sum(c_RIB[i:end-1]))
        end
    end


        # dynamic efficiency factor 
    if tags.dynamic_efficiency == true

        c_T3 = u[tags.idx.a1t : tags.idx.y1t];
        efac = zeros(Float64, 62,1); 

        WC = tags.T_WC * c_T3;
        WB = tags.T_WB * c_T3;
        NC = tags.T_NC * c_T3;
        NN = tags.T_NN * c_T3;

        efac = dynEffficiencyCalculation(WC,WB,NC,NN,EGT);

        tags.efac = efac
    end
        

    # V_EL
    C_vect = reshape(u[1:num_species], 1, num_species)  # Directly assign
    c_Codon = tags.t3_Codon_struc * C_vect'

    V_TL_EL	= zeros(len_sequence-1);

    EGT_inv = inv(EGT)

    # rna_free = u[170]
    # s70_ini = u[171]
    # # this calculation is done according to the analyze_parllel script and is fitting to the description in the Nieß paper
    # q_R_elo[1] = 1 - (sum(c_RIB[1:12]) +s70_ini)/(rna_free+sum(c_RIB[1:12])) 


    for i in 1: len_sequence-2

        j = findfirst(x -> x == String(tags.Seq[i+1]), tags.cod)
        k = findfirst(x -> x == String(tags.Seq[i]), tags.cod)

        if c_Codon[j] == 0
            V_TL_EL[i+1] = 0;  
        else 
            efficiency = tags.efac[k] * min(tags.efac_pos_aa[i], tags.efac_int_sd[i]) ############### The indexing is was altered in comparison to the matlab model from i+1 to i. This is in agreement with the spindler paper and the descriptions in Arnold and Nieß. This is also how it is calculated in analyze_parallel_data.m.
            V_TL_EL[i+1] =  q_R_elo[i] * k_TLEmaxj * c_RIB[i] *  efficiency /(1+ KM_T3j/c_Codon[j]+ KM_EFGGTP*EGT_inv); ########### Indexing of q_R_elo was altered from i+1 to i. Now its same as in analyze_paralle_data data and the description in Nieß paper.  
        end

    end
       
    

    # V_TL_EL     .= V_TL_EL .* q_R_elo; # original version that is like in the mod_seq matlab script

    V_TL_EL_sum = sum(V_TL_EL);

    c_codon_seq = tags.matrix_N * c_Codon
    N = C_vect[1,:] * (1 ./c_codon_seq)';

    # Replace Inf values with 0
    N[isinf.(N)] .= 0
    # Replace NaN values with 0
    N[isnan.(N)] .= 0

    M = zeros(size(N))

    for i in 1:size(N,1)
        for k in 1:size(N,2)
            M[i,k] = tags.alpha_struc[i,k] * N[i,k]
        end
    end


    K_T3 = M*V_TL_EL;

    mtRNA 		= zeros(size(M,1),2);
    M_tRNA 		= hcat(mtRNA,M)
    M_tRNA		= M_tRNA[:, Not(3)]
    M_tRNA		= M_tRNA[:, 1:end-1]

    K_tRNA 		= M_tRNA*V_TL_EL   

    ## translation termination
    te = len_sequence-1 # Number of last coding codon
    # ke = findfirst(x -> x == tags["Seq1"]["Codon"][te], tags["cod"])

    k_RF1_diss_GTP = k_RF1_diss_GTP#*tags.scale.termination

    efficiency_term = 1;
    k_ter_RF1 = 1 / ( 1/(k_RF1_ass*RFt) + 1/(k_RF1_diss_GTP) );
    k_ter_RF3 = 1 / ( 1/(k_RF1_ass*RGT) + 1/(k_RF3_diss_GDP) + 1/(k_GTP_hydr) );

    

    if k_ter_RF1 < k_ter_RF3
        k_ter = k_ter_RF1;
    else
        k_ter = k_ter_RF3;
    end

    V_TE_max = k_ter * RFt   # RFt is RF1/2 total
    V_TL_TE = V_TE_max * efficiency_term / (1 + KM_RK/ c_RIB[te] + KM_RF3GTP/RGT + KM_RK * KM_RF3GTP /(c_RIB[te] * RGT))
        
    if c_RIB[te] == 0 || GTP == 0
        V_TL_TE=0;
    end


    V_RF3_GDP_ass  = k_RF3_GDP_ass  * RF3 * GDP;
    V_RF3_GTP_ass  = k_RF3_GTP_ass  * RF3 * GTP;
    V_RF3_GDP_diss = k_RF3_GDP_diss * RGD;
    V_RF3_GTP_diss = k_RF3_GTP_diss * RGT 

    ## post termination complex recycling
    t_rec = A1 + A2 * 1/RRF + A3 * EGT/RRF + A4 * 1/EGT + A5 * 1/(RRF*EGT); # time/s for recycling according to Borg et al. 2016
    k_rec = 1/t_rec; # rate/s^-1 for recycling according to Borg et al. 2016
    k_rec_min = k_rec*60; # rate/min^-1 for recycling according to Borg et al. 2016
    f_s = k_cat_K_M_G1/(k_RRF * RRF)*(q_RRF/k_cat_K_M_G2*one_q_G2_k_split + EGT)+ one_q_G2_k_split; #EGT hydrolised per splitting event according to Borg et al 2016
        
    ## translation termination - calculation of du 
    tRNA_K = tags.alpha_struc[:,end] * V_TL_TE

    du[1:num_species] .+= tRNA_K;

    rate_RIB[len_sequence-1] += - V_TL_TE

    ######### !!!!!!!!!!! Indexing of output variables needs to be adjusted to single sequence ??????????????? Or is it????????????
    du[tags.idx.PTC] += V_TL_TE - k_rec_min * PTC;	            # PTC
    du[tags.idx.S30] += k_rec_min * PTC;                         # rate of 30S release due to recycling
    du[tags.idx.S50] +=  k_rec_min * PTC;                        # rate of 50S release due to recycling
    du[tags.idx.PRT] = V_TL_TE;                                  # rate of PRT (Protein)
    du[tags.idx.RGT] = -V_TL_TE -V_RF3_GTP_diss +V_RF3_GTP_ass;  # rate of RGT
    du[tags.idx.RGD] =  V_TL_TE -V_RF3_GDP_diss +V_RF3_GDP_ass;  # rate of RGD
    du[tags.idx.RF3] =          +V_RF3_GDP_diss +V_RF3_GTP_diss -V_RF3_GTP_ass -V_RF3_GDP_ass; # rate of RF3

    # translation elongation - calculation of du 

    if RNA == 0 # concentration of free mRNA
        du = du
    else

        # concentration of the ribosome at the specific codon
        rate_RIB[1] = - V_TL_EL[2];  

        for i in 2: len_sequence-2
            rate_RIB[i] = V_TL_EL[i] - V_TL_EL[i+1];	
        end

        rate_RIB[len_sequence-1] = rate_RIB[len_sequence-1] + V_TL_EL[end];

        # concentration of the unloaded tRNA
        du[tags.idx.ta1:tags.idx.ty1] .+= K_tRNA[end-tags.no_of_tRNAs+1:end]

        # concentration of the ternary-complex
        du[1: num_species] .+= - K_T3;
        du[tags.idx.EGT]  += - V_TL_EL_sum - f_s * k_rec_min * PTC;	# Rate of EFG_GTP 
        du[tags.idx.EGD]  += + V_TL_EL_sum + f_s * k_rec_min * PTC;	# Rate of EFG_GDP
        du[tags.idx.RNA] += + V_TL_EL[12];
        du[tags.idx.rna] += - V_TL_EL[12];

    end

    # save RIB in du
    du[idx_first_codon:end] .+= rate_RIB;

    return du
end